# Sweep acquisition

Reads board nodes through the relay mux with the GDM-8341 and appends each reading to a results log. Run it while the thermal chamber steps through its profile, then move the log into the experiment's `data/` folder.

Wiring:
- DMM negative goes to the common line.
- All relay center contacts go to DMM positive. Normally closed contacts are unused.
- Each normally open contact goes to the node being measured.

Opening `RelayMux` opens every relay. The bridge MCU needs `src/instruments/xl9535_relay_mux/firmware/`.

In [ ]:
from datetime import datetime
from pathlib import Path

from gdm8341.driver import Gdm8341
from gdm8341.driver import find_port as find_gdm8341_port
from xl9535_relay_mux.driver import RelayMux
from xl9535_relay_mux.driver import find_port as find_relay_mcu_port
from cryojane_bench.sweep import QuantityType, SweepChannel, run_sweep

# Resistance for a passive run with the boards disconnected (sensor_characterization).
# Voltage for a powered run (heater_characterization).
QUANTITY = QuantityType.RESISTANCE

CHANNELS = [
    SweepChannel("Sensor (0)", QUANTITY, relay_index=0),
    SweepChannel("Heater (0)", QUANTITY, relay_index=1),
    SweepChannel("Sensor (1)", QUANTITY, relay_index=2),
    SweepChannel("Heater (1)", QUANTITY, relay_index=3),
]
output_path = Path(f"results_{datetime.now():%Y%m%d-%H%M%S}.txt")

In [ ]:
# Runs until the kernel is interrupted.
with RelayMux(find_relay_mcu_port()) as mux, Gdm8341(find_gdm8341_port()) as dmm:
    print(dmm.identity, "->", output_path)
    run_sweep(mux, dmm, CHANNELS, output_path)